# Skin Condition Detection — v4
## Key changes from v3
| Problem in v3 | Fix in v4 |
|---|---|
| DINOv2 warmup had only 5K trainable params → never learned properly | Use **Noisy-Student EfficientNet-B2** as solid default |
| All classes imbalanced → model biased | **Balanced dataset built to disk** before training |
| WeightedSampler + FocalLoss on balanced data = redundant | Removed both — simple CrossEntropy + LabelSmoothing |
| Augmentation applied to everything indiscriminately | **Smart augmentation**: only minority classes get synthetic images |
| Normal class (10K) dominated gradients | Downsampled to same target as others |

## Strategy
1. Build a `balanced_dataset/` folder on disk: copy/downsample large classes, augment small ones
2. Train on that balanced folder — no sampler tricks needed
3. Noisy-Student B2 backbone (pretrained on 300M images via self-training)

In [ ]:
# ── COLAB FLAG ────────────────────────────────────────────────────────────
IS_COLAB = False   # <── set True on Google Colab

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Install gdown for Drive downloads
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)

    # ── Download updated dataset ──────────────────────────────────────────
    FILE_ID    = '1ryhBhqynbaSp43ujJ8PXQR1i7RQ8x6wU'
    ZIP_PATH   = '/content/dataset_raw.zip'
    RAW_DIR    = '/content/dataset_raw'

    import os
    if not os.path.exists(ZIP_PATH):
        print('Downloading dataset from Google Drive...')
        subprocess.run(['gdown', '--id', FILE_ID, '-O', ZIP_PATH], check=True)
        print('Download complete.')
    else:
        print('Zip already present, skipping download.')

    import zipfile, pathlib
    if not pathlib.Path(RAW_DIR).exists():
        print('Extracting...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            zf.extractall(RAW_DIR)
        print('Extracted.')

    # Handle single inner folder
    _subdirs = [p for p in pathlib.Path(RAW_DIR).iterdir() if p.is_dir()]
    RAW_DATASET = str(_subdirs[0]) if len(_subdirs) == 1 else RAW_DIR
    BALANCED_DIR = '/content/balanced_dataset'
    OUTPUT_DIR   = '/content/artifacts'

else:
    RAW_DATASET  = '/home/dev/Desktop/cnn/dataset'     # <── your raw dataset path
    BALANCED_DIR = '/home/dev/Desktop/cnn/balanced_dataset'
    OUTPUT_DIR   = '/home/dev/Desktop/cnn/artifacts'

import os
print('Raw dataset  :', RAW_DATASET)
print('Balanced dir :', BALANCED_DIR)
print('Output dir   :', OUTPUT_DIR)

In [ ]:
# !pip install -q timm torchinfo scikit-learn seaborn

import os, json, time, copy, math, random, shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import datasets, transforms, models

import timm

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except ImportError:
    HAS_TORCHINFO = False

try:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  STEP 1 — Build a balanced dataset on disk
#
#  Rules:
#    - Classes with count > TARGET : randomly copy TARGET images (no aug)
#    - Classes with count < TARGET : copy all originals, then generate
#                                    augmented images until reaching TARGET
#    - Result: every class folder has exactly TARGET images
# ════════════════════════════════════════════════════════════════════════════

TARGET_PER_CLASS = 1800   # <── adjust if you want more/fewer total images
SEED             = 42
REBUILD          = False  # set True to regenerate even if balanced_dataset exists

random.seed(SEED)
np.random.seed(SEED)

# ── Augmentation pipeline for synthetic image generation (saved as JPEGs) ──
# Only spatial + colour transforms — NO normalization (we save raw pixels)
_aug_for_save = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.70, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.06),
    transforms.RandomGrayscale(p=0.04),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
])

def make_balanced_dataset(raw_root, balanced_root, target, rebuild=False, seed=42):
    raw_root      = Path(raw_root)
    balanced_root = Path(balanced_root)

    if balanced_root.exists() and not rebuild:
        # Check if it looks complete
        existing = {p.name: len(list(p.glob('*.*')))
                    for p in balanced_root.iterdir() if p.is_dir()}
        if existing and all(v >= target * 0.95 for v in existing.values()):
            print(f'Balanced dataset already exists at {balanced_root}')
            print('  Counts:', {k: v for k, v in sorted(existing.items())})
            print('  Set REBUILD=True to regenerate.')
            return balanced_root

    print(f'Building balanced dataset → {balanced_root}')
    print(f'  Target per class: {target}')

    class_dirs = sorted([p for p in raw_root.iterdir() if p.is_dir()])
    if not class_dirs:
        raise RuntimeError(f'No class sub-folders found in {raw_root}')

    report_rows = []
    for cls_dir in class_dirs:
        cls_name = cls_dir.name
        images   = sorted([p for p in cls_dir.iterdir()
                           if p.suffix.lower() in ('.jpg','.jpeg','.png','.webp','.bmp')])
        n_orig   = len(images)
        if n_orig == 0:
            print(f'  WARNING: {cls_name} has 0 images, skipping.')
            continue

        out_dir = balanced_root / cls_name
        out_dir.mkdir(parents=True, exist_ok=True)

        rng = random.Random(seed + hash(cls_name) % 10000)
        rng.shuffle(images)

        if n_orig >= target:
            # ── Downsample: copy first `target` after shuffle ─────────────
            selected = images[:target]
            for i, src in enumerate(selected):
                dst = out_dir / f'orig_{i:05d}{src.suffix.lower()}'
                shutil.copy2(src, dst)
            n_copied, n_aug = target, 0

        else:
            # ── Copy all originals ────────────────────────────────────────
            for i, src in enumerate(images):
                dst = out_dir / f'orig_{i:05d}{src.suffix.lower()}'
                shutil.copy2(src, dst)

            # ── Generate augmented images until we reach target ───────────
            n_needed = target - n_orig
            aug_idx  = 0
            pool     = images  # always augment from originals only

            pbar = tqdm(total=n_needed, desc=f'  Augmenting {cls_name:<18}', leave=False)
            while aug_idx < n_needed:
                src = pool[aug_idx % len(pool)]
                try:
                    img = Image.open(src).convert('RGB')
                    aug = _aug_for_save(img)
                    dst = out_dir / f'aug_{aug_idx:05d}.jpg'
                    aug.save(dst, quality=92)
                    aug_idx += 1
                    pbar.update(1)
                except Exception as e:
                    print(f'    Skipped {src.name}: {e}')
            pbar.close()
            n_copied, n_aug = n_orig, n_needed

        actual = len(list(out_dir.glob('*.*')))
        report_rows.append({'class': cls_name, 'original': n_orig,
                            'copied': n_copied, 'augmented': n_aug,
                            'total_in_folder': actual})
        print(f'  {cls_name:<22} orig={n_orig:5d}  '
              f'copied={n_copied:5d}  aug={n_aug:5d}  → {actual:5d}')

    print(f'Done. Total classes: {len(report_rows)}')
    return balanced_root, pd.DataFrame(report_rows)


balanced_path, balance_report = make_balanced_dataset(
    RAW_DATASET, BALANCED_DIR, TARGET_PER_CLASS, rebuild=REBUILD, seed=SEED
)
display(balance_report)

# Verify
total_images = sum(balance_report['total_in_folder'])
print(f'\nTotal balanced images : {total_images}')
print(f'Expected             : {len(balance_report) * TARGET_PER_CLASS}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
@dataclass
class Config:
    dataset_root: str = BALANCED_DIR   # ← now points to balanced folder
    output_dir:   str = OUTPUT_DIR
    run_name:     str = 'skin_v4_ns_b2'

    img_size:    int   = 260    # EfficientNet-B2 native resolution
    batch_size:  int   = 32
    num_workers: int   = 2

    # Training
    epochs:               int   = 45
    lr:                   float = 2e-4
    weight_decay:         float = 4e-4
    label_smoothing:      float = 0.08
    warmup_epochs:        int   = 5    # longer warmup than v3
    freeze_backbone_epochs: int = 5    # head-only for 5 epochs before unfreezing

    # Grad accumulation: eff_batch = batch_size × accum_steps
    accum_steps: int = 2

    # Augmentation (MixUp + CutMix alternated)
    mixup_alpha:  float = 0.3
    cutmix_alpha: float = 1.0
    cutmix_prob:  float = 0.5

    # SWA
    use_swa:   bool  = True
    swa_start: int   = 22
    swa_lr:    float = 4e-5

    # Early stopping
    patience:  int   = 10
    min_delta: float = 1e-4

    seed: int = 42


cfg = Config()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

RUN_DIR = Path(cfg.output_dir) / cfg.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run dir  :', RUN_DIR)
print('Img size :', cfg.img_size)
print('Device   :', DEVICE)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Training augmentation (applied live during training on already-balanced data)
# More conservative than v3 because the dataset is already diverse via offline aug
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.10),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.04),
    transforms.RandAugment(num_ops=2, magnitude=6),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), value='random'),
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(cfg.img_size * 1.14)),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# TTA views for inference
tta_tfms = [
    eval_tfms,
    transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.14)),
        transforms.CenterCrop(cfg.img_size),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.22)),
        transforms.CenterCrop(cfg.img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
]
print('Transforms ready.')

In [ ]:
base_dataset = datasets.ImageFolder(cfg.dataset_root)
class_names  = base_dataset.classes
num_classes  = len(class_names)
print('Classes    :', class_names)
print('Num classes:', num_classes)

with open(RUN_DIR / 'class_to_idx.json', 'w') as f:
    json.dump(base_dataset.class_to_idx, f, indent=2)

# ── Stratified 75/15/10 split ─────────────────────────────────────────────
targets = np.array(base_dataset.targets)
by_class = defaultdict(list)
for idx, y in enumerate(targets):
    by_class[int(y)].append(idx)

train_idx, val_idx, test_idx = [], [], []
for cls, idxs in by_class.items():
    idxs = np.array(idxs)
    rng  = np.random.default_rng(cfg.seed + cls)
    rng.shuffle(idxs)
    n       = len(idxs)
    n_train = max(1, int(round(n * 0.75)))
    n_val   = max(1, int(round(n * 0.15)))
    n_test  = n - n_train - n_val
    if n_test < 1:
        n_train -= (1 - n_test)
        n_test   = 1
    train_idx.extend(idxs[:n_train].tolist())
    val_idx.extend(  idxs[n_train:n_train + n_val].tolist())
    test_idx.extend( idxs[n_train + n_val:n_train + n_val + n_test].tolist())

print(f'Train/Val/Test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}')

train_ds = Subset(datasets.ImageFolder(cfg.dataset_root, transform=train_tfms), train_idx)
val_ds   = Subset(datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms),  val_idx)
test_ds  = Subset(datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms),  test_idx)

train_targets = targets[train_idx]
val_targets   = targets[val_idx]
test_targets  = targets[test_idx]

dist_df = pd.DataFrame({
    'class': class_names,
    'train': np.bincount(train_targets, minlength=num_classes),
    'val':   np.bincount(val_targets,   minlength=num_classes),
    'test':  np.bincount(test_targets,  minlength=num_classes),
})
display(dist_df)

# ── DataLoaders (no WeightedSampler needed — data is balanced) ────────────
_nw = cfg.num_workers
_pm = (DEVICE == 'cuda')
_pw = (_nw > 0)

train_loader = DataLoader(train_ds,  batch_size=cfg.batch_size, shuffle=True,
                          num_workers=_nw, pin_memory=_pm, persistent_workers=_pw)
val_loader   = DataLoader(val_ds,    batch_size=cfg.batch_size, shuffle=False,
                          num_workers=_nw, pin_memory=_pm, persistent_workers=_pw)
test_loader  = DataLoader(test_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=_nw, pin_memory=_pm, persistent_workers=_pw)

print(f'Loaders: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)} batches')

In [ ]:
# ── Model: EfficientNet-B2 Noisy Student ─────────────────────────────────
#
# Why Noisy-Student over plain ImageNet weights?
#   • Pretrained on JFT-300M (300M images) using iterative self-training
#   • Student model trained WITH noise (dropout, stochastic depth, RandAugment)
#     during distillation → learns more robust, generalisable features
#   • Benchmark: +2-3% on ImageNet vs standard weights, same architecture
#   • For skin: the richer feature space means less training data needed
#     to separate visually similar classes (Eczema vs Rosacea, etc.)
#
# Head: simple — avoids the memorisation trap of the v2 512-layer head

backbone = timm.create_model(
    'tf_efficientnet_b2.ns_jft_in1k',
    pretrained=True,
    num_classes=0,   # strip head
)
in_features = backbone.num_features   # 1408 for B2

class SkinModel(nn.Module):
    def __init__(self, backbone, in_features, num_classes):
        super().__init__()
        self.backbone = backbone
        # GeM pooling generalises better than avg-pool for texture recognition
        self.gem_p  = nn.Parameter(torch.ones(1) * 3.0)
        self.head   = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=0.40),
            nn.Linear(in_features, num_classes),
        )

    def gem_pool(self, x):
        # x: (B, C, H, W)  →  (B, C, 1, 1)
        p = self.gem_p.clamp(min=1.0)
        return F.avg_pool2d(x.clamp(min=1e-6).pow(p),
                            (x.size(-2), x.size(-1))).pow(1.0 / p)

    def forward(self, x):
        feat = self.backbone.forward_features(x)   # (B, C, H, W)
        feat = self.gem_pool(feat).flatten(1)       # (B, C)
        return self.head(feat)


model = SkinModel(backbone, in_features, num_classes).to(DEVICE)

# Freeze backbone for warmup
for p in backbone.parameters():
    p.requires_grad = False

print('Model built.')
print(model.head)

In [ ]:
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print('=' * 68)
print(f'  BACKBONE   :  EfficientNet-B2 Noisy-Student (tf_efficientnet_b2.ns_jft_in1k)')
print(f'  IMG SIZE   :  {cfg.img_size}×{cfg.img_size}')
print(f'  CLASSES    :  {num_classes}  →  {class_names}')
print(f'  DATASET    :  {len(train_idx)+len(val_idx)+len(test_idx):,} total  '
      f'(balanced: {TARGET_PER_CLASS}/class)')
print('-' * 68)
print(f'  Total params     : {total_p:>12,}  ({total_p/1e6:.2f} M)')
print(f'  Trainable now    : {trainable_p:>12,}  ({trainable_p/1e6:.2f} M)'
      '  ← head only during warmup')
print(f'  Frozen (warmup)  : {frozen_p:>12,}  ({frozen_p/1e6:.2f} M)')
print(f'  Head in features : {in_features}  (EfficientNet-B2 feature dim)')
print('-' * 68)
print(f'  freeze_backbone_epochs : {cfg.freeze_backbone_epochs}')
print(f'  warmup_epochs          : {cfg.warmup_epochs}')
print(f'  Effective batch size   : {cfg.batch_size} × {cfg.accum_steps} = '
      f'{cfg.batch_size*cfg.accum_steps}')
print(f'  SWA                    : starts epoch {cfg.swa_start}, lr={cfg.swa_lr}')
print(f'  MixUp α                : {cfg.mixup_alpha}')
print(f'  CutMix α               : {cfg.cutmix_alpha}  (prob={cfg.cutmix_prob})')
print('=' * 68)

# Layer type breakdown
layer_counts = {}
for _, m in model.named_modules():
    t = type(m).__name__
    layer_counts[t] = layer_counts.get(t, 0) + 1
print('\nLayer type breakdown:')
for t, c in sorted(layer_counts.items(), key=lambda x: -x[1]):
    if t not in ('Sequential', 'SkinModel') and c > 0:
        print(f'  {t:<35} × {c}')

if HAS_TORCHINFO:
    print()
    torchinfo_summary(model,
                      input_size=(2, 3, cfg.img_size, cfg.img_size),
                      device=DEVICE,
                      col_names=['input_size','output_size','num_params','trainable'],
                      depth=2, verbose=1)
else:
    print('\n[pip install torchinfo for full per-layer table]')

In [ ]:
# Simple CrossEntropy + label smoothing — no class weights needed (balanced data!)
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
print('Loss:', criterion)

# AdamW — head only initially
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.lr, weight_decay=cfg.weight_decay,
)

def lr_lambda(epoch):
    '''Linear warmup → cosine decay with 5% floor.'''
    if epoch < cfg.warmup_epochs:
        return (epoch + 1) / max(1, cfg.warmup_epochs)
    t = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * t)))

scheduler    = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
swa_model    = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=cfg.swa_lr, anneal_epochs=5)
swa_active   = False
scaler       = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

print(f'Optimiser: AdamW lr={cfg.lr}  wd={cfg.weight_decay}')
print(f'Scheduler: warmup {cfg.warmup_epochs}ep → cosine decay')

In [ ]:
def mixup_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cut_h = int(H * math.sqrt(1.0 - lam))
    cut_w = int(W * math.sqrt(1.0 - lam))
    cx, cy = random.randint(0, W), random.randint(0, H)
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, W)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, H)
    mixed = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (y2 - y1) * (x2 - x1) / float(H * W)
    return mixed, y, y[idx], lam_actual

def mixed_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

print('MixUp / CutMix ready.')

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None,
              mixup_alpha=0.0, cutmix_alpha=0.0, cutmix_prob=0.5, accum_steps=1):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, correct, total = 0.0, 0, 0
    yt_all, yp_all = [], []

    pbar = tqdm(enumerate(loader), total=len(loader),
                desc='train' if is_train else 'eval ', leave=False)

    for step, (x, y) in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if is_train and (mixup_alpha > 0 or cutmix_alpha > 0):
            if random.random() < cutmix_prob and cutmix_alpha > 0:
                x, ya, yb, lam = cutmix_data(x, y, cutmix_alpha)
            else:
                x, ya, yb, lam = mixup_data(x, y, mixup_alpha)
            use_mix = True
        else:
            ya, yb, lam, use_mix = y, y, 1.0, False

        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x)
            loss   = (mixed_loss(criterion, logits, ya, yb, lam)
                      if use_mix else criterion(logits, ya))
            loss   = loss / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if (step + 1) % accum_steps == 0 or step + 1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * accum_steps * x.size(0)
        correct    += (preds == ya).sum().item()
        total      += x.size(0)

        yt_all.append(ya.detach().cpu())
        yp_all.append(preds.detach().cpu())
        pbar.set_postfix(loss=f'{total_loss/total:.4f}', acc=f'{correct/total:.4f}')

    yt = torch.cat(yt_all).numpy() if yt_all else np.array([])
    yp = torch.cat(yp_all).numpy() if yp_all else np.array([])
    return total_loss / max(total, 1), correct / max(total, 1), yt, yp


def per_class_acc(yt, yp, n):
    return {c: float((yp[yt==c]==c).mean()) if (yt==c).sum() > 0 else float('nan')
            for c in range(n)}


def plot_history(history_df, swa_start=None, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history_df.epoch, history_df.train_loss, label='Train')
    axes[0].plot(history_df.epoch, history_df.val_loss,   label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history_df.epoch, history_df.train_acc, label='Train')
    axes[1].plot(history_df.epoch, history_df.val_acc,   label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend()
    axes[2].plot(history_df.epoch, history_df.lr)
    axes[2].set_title('LR'); axes[2].set_yscale('log')
    if swa_start:
        for ax in axes:
            ax.axvspan(swa_start, history_df.epoch.max(), alpha=0.07,
                       color='green', label='SWA')
    for ax in axes:
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


def plot_confusion(yt, yp, names, save_path=None):
    if not HAS_SKLEARN:
        return
    cm  = confusion_matrix(yt, yp)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


print('Helper functions defined.')

In [ ]:
history            = []
best_val_acc       = -np.inf
best_state         = None
patience_counter   = 0
backbone_unfrozen  = False
swa_active_flag    = False

start = time.time()

for epoch in range(1, cfg.epochs + 1):

    # ── Unfreeze backbone ────────────────────────────────────────────────
    if epoch == cfg.freeze_backbone_epochs + 1 and not backbone_unfrozen:
        for p in backbone.parameters():
            p.requires_grad = True
        backbone_unfrozen = True

        backbone_params = list(backbone.parameters())
        backbone_ids    = {id(p) for p in backbone_params}
        head_params     = [p for p in model.parameters() if id(p) not in backbone_ids]

        # Differential LR: backbone 10× lower than head
        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': cfg.lr * 0.08},
            {'params': head_params,     'lr': cfg.lr * 0.4},
        ], weight_decay=cfg.weight_decay)
        scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        swa_model  = AveragedModel(model)

        tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
        tot = sum(p.numel() for p in model.parameters())
        print(f'✓ Backbone unfrozen | trainable: {tp/1e6:.2f}M / {tot/1e6:.2f}M')

    # ── Activate SWA ─────────────────────────────────────────────────────
    if cfg.use_swa and epoch == cfg.swa_start and backbone_unfrozen and not swa_active_flag:
        swa_active_flag = True
        print(f'✓ SWA activated at epoch {epoch}')

    # ── Train + Val ───────────────────────────────────────────────────────
    tr_loss, tr_acc, _, _ = run_epoch(
        model, train_loader, criterion, optimizer, scaler,
        mixup_alpha=cfg.mixup_alpha, cutmix_alpha=cfg.cutmix_alpha,
        cutmix_prob=cfg.cutmix_prob, accum_steps=cfg.accum_steps,
    )
    va_loss, va_acc, yv, pv = run_epoch(model, val_loader, criterion)

    if swa_active_flag:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    lr_now = optimizer.param_groups[0]['lr']
    row    = dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                  val_loss=va_loss, val_acc=va_acc, lr=lr_now,
                  swa=swa_active_flag)
    history.append(row)

    print(f'Epoch {epoch:02d}/{cfg.epochs} | '
          f'train {tr_loss:.4f}/{tr_acc:.4f} | '
          f'val {va_loss:.4f}/{va_acc:.4f} | '
          f'lr={lr_now:.2e}' + (' [SWA]' if swa_active_flag else ''))

    # ── Checkpoint ───────────────────────────────────────────────────────
    if (va_acc - best_val_acc) > cfg.min_delta:
        best_val_acc = va_acc
        best_state = {
            'model':        copy.deepcopy(model.state_dict()),
            'epoch':        epoch,
            'val_acc':      va_acc,
            'config':       asdict(cfg),
            'class_names':  class_names,
            'class_to_idx': base_dataset.class_to_idx,
        }
        torch.save(best_state, RUN_DIR / 'best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= cfg.patience:
        print(f'Early stopping at epoch {epoch}.')
        break

# ── SWA finalisation ──────────────────────────────────────────────────────
if swa_active_flag:
    print('\nUpdating SWA BatchNorm statistics...')
    update_bn(train_loader, swa_model, device=DEVICE)
    _, swa_va, _, _ = run_epoch(swa_model, val_loader, criterion)
    print(f'SWA val acc: {swa_va:.4f}  (best single: {best_val_acc:.4f})')
    if swa_va > best_val_acc:
        print('SWA is better — saving as best checkpoint.')
        torch.save({
            'model':        swa_model.module.state_dict(),
            'epoch':        epoch,
            'val_acc':      swa_va,
            'config':       asdict(cfg),
            'class_names':  class_names,
            'class_to_idx': base_dataset.class_to_idx,
            'swa':          True,
        }, RUN_DIR / 'best_model.pt')
        best_val_acc = swa_va

elapsed = time.time() - start
print(f'\nDone in {elapsed/60:.1f} min | Best val acc: {best_val_acc:.4f}')

history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / 'train_history.csv', index=False)
display(history_df.tail(10))

In [ ]:
plot_history(history_df,
             swa_start=(cfg.swa_start if swa_active_flag else None),
             save_path=RUN_DIR / 'training_curves.png')

# Overfitting diagnostic
last = history_df.iloc[-1]
gap  = last.train_acc - last.val_acc
print(f'--- Overfitting check ---')
print(f'Train acc: {last.train_acc:.4f}  Val acc: {last.val_acc:.4f}  Gap: {gap:.4f}')
print(f'Loss ratio (val/train): {last.val_loss / max(last.train_loss, 1e-8):.2f}×')
if   gap < 0.05: print('✅ Good generalisation (gap < 5%)')
elif gap < 0.10: print('⚠️  Mild overfit (gap 5-10%)')
else:            print('❌ Significant overfit (gap > 10%)')

In [ ]:
# Load best checkpoint
ckpt = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]} | val_acc={ckpt["val_acc"]:.4f}'
      + (' (SWA)' if ckpt.get('swa') else ''))

test_loss, test_acc, yt, yp = run_epoch(model, test_loader, criterion)
print(f'\nTest loss: {test_loss:.4f} | Test acc: {test_acc:.4f}')

# Per-class table
pc = per_class_acc(yt, yp, num_classes)
pc_df = pd.DataFrame({
    'class':    class_names,
    'n_test':   np.bincount(test_targets, minlength=num_classes),
    'acc':      [round(pc[i], 4) for i in range(num_classes)],
})
display(pc_df.sort_values('acc'))
pc_df.to_csv(RUN_DIR / 'test_per_class_accuracy.csv', index=False)

if HAS_SKLEARN:
    print('\nClassification Report:')
    print(classification_report(yt, yp, target_names=class_names, digits=3))
    pd.DataFrame(
        classification_report(yt, yp, target_names=class_names, digits=3, output_dict=True)
    ).T.to_csv(RUN_DIR / 'classification_report.csv')

plot_confusion(yt, yp, class_names, save_path=RUN_DIR / 'confusion_matrix.png')

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names':      class_names,
    'class_to_idx':     base_dataset.class_to_idx,
    'img_size':         cfg.img_size,
    'normalize_mean':   MEAN,
    'normalize_std':    STD,
    'architecture':     'tf_efficientnet_b2.ns_jft_in1k',
    'val_acc':          float(best_val_acc),
    'test_acc':         float(test_acc),
}, RUN_DIR / 'model_inference.pt')

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

print('Saved artifacts:')
for p in sorted(RUN_DIR.glob('*')):
    print(f'  {p.name:<44} {p.stat().st_size/1024:>8.1f} KB')

In [ ]:
if IS_COLAB:
    from google.colab import files
    import zipfile

    bundle = f'/content/skin_v4_artifacts.zip'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(RUN_DIR.glob('*')):
            zf.write(p, arcname=p.name)
    print(f'Downloading {bundle} ...')
    files.download(bundle)
else:
    print('Artifacts at:', RUN_DIR)

In [ ]:
def predict_image(image_path: str, checkpoint_path: str = None,
                  use_tta: bool = True) -> dict:
    '''Single-image inference with optional 3-view TTA.'''
    checkpoint_path = checkpoint_path or str(RUN_DIR / 'model_inference.pt')
    blob = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

    # Rebuild model from checkpoint
    _bb = timm.create_model(blob['architecture'], pretrained=False, num_classes=0)
    _in = _bb.num_features
    _m  = SkinModel(_bb, _in, len(blob['class_names'])).to(DEVICE)
    _m.load_state_dict(blob['model_state_dict'])
    _m.eval()

    im    = Image.open(image_path).convert('RGB')
    views = tta_tfms if use_tta else [eval_tfms]
    probs = []

    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
        for tfm in views:
            x = tfm(im).unsqueeze(0).to(DEVICE)
            probs.append(torch.softmax(_m(x), dim=1)[0])

    avg   = torch.stack(probs).mean(0)
    conf, idx = avg.max(0)
    names = blob['class_names']
    top3  = sorted(enumerate(avg.cpu().tolist()), key=lambda z: -z[1])[:3]

    return {
        'pred_class': names[idx.item()],
        'confidence': round(float(conf), 4),
        'tta_views':  len(views),
        'top3':       [(names[i], round(p, 4)) for i, p in top3],
    }

# Example:
# print(predict_image('/path/to/skin_image.jpg'))